In [ ]:
%pip install neuron

from neuron.units import mV,ms,um # unit definition
from matplotlib import pyplot
import numpy as np
import itertools
from neuron import h, gui, rxd # chemical dynamic
import plotly
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.interpolate import interp1d
from scipy.optimize import differential_evolution
from neuron.units import mV, ms, um  # NEURON-defined units


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# All functions
h.load_file("stdlib.hoc")
h.load_file("import3d.hoc")

def load_neuron_model(hoc_file_path):
    """
    Load a neuron model from a hoc file, set up the NEURON environment,
    and allow for flexible naming of the NEURON object.
    """
    for sec in h.allsec():
        h.delete_section(sec=sec)

    h.load_file(hoc_file_path)
    ps = h.PlotShape(False)
    ps.plot(plotly).show()

    neuron_obj = h
    return neuron_obj

def record_segment(inj_site, recor_site, current = -1, duration= 3250, stim_dur = 1500, delay = 250, init_vm=-65, delta_t=0.05):
    """
    Record membrane potential at a specific segment in the neuron.
    """

    # Set simulation parameters
    h.tstop = duration  # Total duration in ms
    h.v_init = init_vm  # Initial V_m (mV)
    h.dt = delta_t      # Time step for simulation (ms)

    # Create current clamp at the specified segment
    stim = h.IClamp(inj_site)
    stim.delay = delay      # Start of stimulus (ms)
    stim.dur = stim_dur  # Duration of stimulus (ms)
    stim.amp = current   # Amplitude of stimulus (nA)

    # Record membrane potential and time
    v_vec = h.Vector()  # V_m vector
    t_vec = h.Vector()  # Time vector
    v_vec.record(recor_site._ref_v)
    t_vec.record(h._ref_t)

    # Initialize and run the simulation
    h.tstop = duration  # Total duration in ms
    h.finitialize(h.v_init)
    h.continuerun(h.tstop)

    return v_vec, t_vec

def fit_passive_parameters(exp_vol, exp_t, inj_site, recor_site, stim_dur, current, init_vm = -65):
    """
    fit Cm, R_a, R_m for the model

    """
    delay_t =  np.round (abs (exp_t[0]) * 1000, decimals = 6) # convert to ms
    exp_t =  np.round ((exp_t + abs (exp_t[0])) * 1000, decimals = 6)  # shift to all positive
    current_end_time = delay_t + stim_dur  # End time of current injection
    t_fit_end = current_end_time
    t_fit_start = current_end_time - 1000

    exp_indices = np.where((exp_t >= t_fit_start) & (exp_t <= t_fit_end))[0]
    exp_v = exp_vol[exp_indices]
    exp_t_window = exp_t[exp_indices]

    # Check if the experimental data is sampled at constant intervals
    delta_t_exp = np.diff(exp_t_window)
    if not np.allclose(delta_t_exp, delta_t_exp[0]):
        print("Experimental data is not sampled at constant intervals. Cannot proceed without interpolation.")
        return

    delta_t_sim = np.round(delta_t_exp[0], decimals = 2) # Set simulation time step to match experimental sampling interval
    N_data = len(exp_t_window)  # Number of data points in fitting window
    print (N_data)

    mse_vec = []
    # Objective function to minimize
    def objective_function(params):
        Ra, cm, Rm = params
        print (f"Trying - Ra: {Ra}, cm: {cm}, g_pas: {1/Rm}")
        # Update model
        for sec in h.allsec():
            sec.insert('pas')
            sec.e_pas = init_vm  # fixed passive Voltage
            sec.Ra = Ra
            sec.cm = cm
            sec.g_pas = np.round (1/Rm, decimals = 5)

        # Simualtion to record voltage and interpolate to exp data
        sim_t = 1020
        duration = sim_t + delay_t + 10 # padding
        v_vec, t_vec = record_segment(inj_site, recor_site, current=current,
                                      duration=duration, stim_dur = sim_t,
                                      delay = delay_t, init_vm=init_vm, delta_t=delta_t_sim)
        v_vec = np.array(v_vec)
        t_vec = np.array(t_vec)

        # Only fit a 1000ms window during injection
        temp = int((sim_t + delay_t - 1000)/delta_t_sim)
        sim_v = v_vec[temp:N_data + temp]
        mse = np.mean((sim_v - exp_v) ** 2)
        mse_vec.append (mse)
        print (f"MSE: {mse}")
        return mse

    # Bounds for parameters
    bounds = [(58,62), (0.8, 1.2), (4000, 5000)]  # Ra, cm, R_M

    # Optimization function
    result = differential_evolution(objective_function, bounds, maxiter=10)

    Ra_opt, cm_opt, Rm_opt = result.x
    print("Optimized Ra:", Ra_opt)
    print("Optimized cm:", cm_opt)
    print("Optimized Rm:", Rm_opt)

    return result, exp_t, exp_vol, mse_vec

def fitExponential (exp_curve):
    # fit an exponential to the first segments of the trace
    


In [ ]:
# load neuron
hocpath = "LGMD_Complete_Construction.hoc"
LGMD1 = load_neuron_model(hocpath)

In [3]:
# based on Peron, 2007
V_m = -65        # Resting membrane potential in mV
R_m = 5000       # Specific membrane resistivity in Ohm·cm²
C_m = 0.7          # Specific membrane capacitance in µF/cm² (NOT CORRECT, Should be <= 1)
#tau_m = 6.6      # Membrane time constant in ms
R_a = 57         # Axial resistance in Ohm·cm
G_pas = 1 / R_m  # Passive leaky conductance in S/cm²
# Input resistance is resistance at the point of current injection

# Insert passive properties into all sections and current injection
for sec in h.allsec():
    sec.insert('pas')        # Insert passive mechanism
    sec.Ra = R_a             # Axial resistivity (Ohm cm)
    sec.cm = C_m             # Membrane capacitance (uF/cm^2)
    sec.g_pas = G_pas        # Leak conductance (S/cm^2)
    sec.e_pas = V_m          # Leak reversal potential (mV)

def record_segment(inj_site, recor_site, current = -1, duration= 3250, stim_dur = 1500, delay = 250, init_vm=-65, delta_t=0.05):
    """
    Record and plot the membrane potential at a specific segment in the neuron.

    Parameters:
    - inj_site, recor_site (e.g., soma[0](0.5)).
    - duration: Total simulation time in ms
    - init_vm: Initial membrane potential
    """

    # Set simulation parameters
    h.tstop = duration  # Total duration in ms
    h.v_init = init_vm  # Initial V_m (mV)
    h.dt = delta_t      # Time step for simulation (ms)

    # Create current clamp at the specified segment
    stim = h.IClamp(inj_site)
    stim.delay = delay      # Start of stimulus (ms)
    stim.dur = stim_dur  # Duration of stimulus (ms)
    stim.amp = current   # Amplitude of stimulus (nA)

    # Record membrane potential and time
    v_vec = h.Vector()  # V_m vector
    t_vec = h.Vector()  # Time vector
    v_vec.record(recor_site._ref_v)
    t_vec.record(h._ref_t)

    # Initialize and run the simulation
    h.tstop = duration  # Total duration in ms
    h.finitialize(h.v_init)
    h.continuerun(h.tstop)

    return v_vec, t_vec

In [ ]:
# Run current injection experiment with
current_values = [-2,-4,-6]  # nA

all_voltages = []
all_times = []
inj_site = LGMD1.FieldA[809](0.5) # base of field A
record_site = LGMD1.FieldA[809](0.5)

# Loop through each current value
for current in current_values:

    v_vec, t_vec = record_segment(inj_site, record_site, current=current)

    # Store the recorded voltage and time vectors
    all_voltages.append(np.array(v_vec))
    all_times.append(np.array(t_vec))

# Plot the results on same graph
plt.figure(figsize=(10, 6))
for i, current in enumerate(current_values):
    plt.plot(all_times[i], all_voltages[i], label=f"Current = {current} nA")

plt.xlabel("Time (ms)")
plt.ylabel("Membrane Potential (mV)")
plt.title("Membrane Potential for Different Current Injections")
plt.legend()
plt.show()